In [1]:
import stackview as sv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import os
from numpy import linalg as lg

In [2]:
a = np.array([[4,3,-8],
          [3, 3, -5],
          [9, 8, -16]])
b = np.array([7, 5, 15])
lg.solve(a, b)

array([-1.,  1., -1.])

In [25]:
def read_raw_image(filepath, dtype, shape = (512, 512)):
    data = np.fromfile(filepath, dtype=dtype)
    return data.reshape(shape)

In [26]:
def normalize_img(img):
    img = img.astype(np.float32)  # ensure float
    img_norm = (img - img.min()) / (img.max() - img.min())
    return img_norm

In [27]:
def read_raw_scan(scan, height = 512, width = 512):
    height = height
    width = width
    dtype = np.uint16
    folder = "../../raw/" + scan
    files = os.listdir('../../raw/' + scan)
    files = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
    slices = []

    for f in files:
        if f.endswith(".raw"):
            img = read_raw_image(
                os.path.join(folder, f),
                shape=(height, width),
                dtype=dtype
            )
            img = normalize_img(img)
            slices.append(img)

    volume = np.stack(slices, axis=0)
    return volume

def read_alt_scan(scan, height = 512, width = 512):
    height = height
    width = width
    dtype = np.uint16
    folder = "../../altered/" + scan
    files = os.listdir('../../altered/' + scan)
    files = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
    slices = []

    for f in files:
        if f.endswith(".raw"):
            img = read_raw_image(
                os.path.join(folder, f),
                shape=(height, width),
                dtype=np.float32
            )
            img = normalize_img(img)
            slices.append(img)

    volume = np.stack(slices, axis=0)
    return volume

In [28]:
def create_side_by_side(vol_1, vol_2):
    side_by_side = []

    for orig, den in zip(vol_1, vol_2):
        combined = np.concatenate([orig, den], axis=1)  # horizontal stack
        side_by_side.append(combined)

    side_by_side = np.stack(side_by_side, axis=0)
    return side_by_side

In [29]:
base_raw_path = '../../raw/'
base_alt_path = '../../altered/'
raw_scan_dirs = sorted(os.listdir(base_raw_path))
raw_scan_dirs.pop()
alt_scan_dirs = sorted(os.listdir(base_alt_path))

In [31]:
def write_manipulated_to_file(volume, scan, verbose = False):
    num = volume.shape[0]
    base_file = '../../resnet_20/' + scan + '/'
    os.makedirs(base_file, exist_ok=True)
    if verbose == True:
        for i in tqdm(range(num)):
            file = '../../resnet_20/' + scan + '/' + f'{scan}_{i+1:03d}.raw'
            volume[i,:,:].tofile(file)
    else:
        for i in range(num):
            file = '../../resnet_20/' + scan + '/' + f'{scan}_{i+1:03d}.raw'
            volume[i,:,:].tofile(file)

In [32]:
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SliceDataset(Dataset):
    def __init__(self, X, Y):
        # Normalize to [0,1] if not already
        self.X = X.astype('float32') / X.max()
        self.Y = Y.astype('float32') / Y.max()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32).unsqueeze(0)  # (1,H,W)
        y = torch.tensor(self.Y[idx], dtype=torch.float32).unsqueeze(0)
        return x, y

print(device)

cuda


In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# Residual Block (with optional dilation)
# -------------------------
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation)
        self.norm1 = nn.InstanceNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=dilation, dilation=dilation)
        self.norm2 = nn.InstanceNorm2d(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = F.leaky_relu(self.norm1(self.conv1(x)), 0.1, inplace=True)
        out = self.norm2(self.conv2(out))
        return F.leaky_relu(out + identity, 0.1, inplace=True)


# -------------------------
# Down Block
# -------------------------
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x_res = self.res(x)
        x_pool = self.pool(x_res)
        return x_res, x_pool


# -------------------------
# Up Block
# -------------------------
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, out_ch, 1)
        )
        self.res = ResBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)


# -------------------------
# Residual U-Net for Streak Removal
# -------------------------
class StreakResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=32):
        super().__init__()

        # Encoder
        self.d1 = DownBlock(in_channels, base_filters)
        self.d2 = DownBlock(base_filters, base_filters*2)
        self.d3 = DownBlock(base_filters*2, base_filters*4)
        self.d4 = DownBlock(base_filters*4, base_filters*8)

        # Bottleneck with dilation to capture long streaks
        self.bottleneck = ResBlock(base_filters*8, base_filters*16, dilation=2)

        # Decoder
        self.u4 = UpBlock(base_filters*16, base_filters*8, base_filters*8)
        self.u3 = UpBlock(base_filters*8, base_filters*4, base_filters*4)
        self.u2 = UpBlock(base_filters*4, base_filters*2, base_filters*2)
        self.u1 = UpBlock(base_filters*2, base_filters, base_filters)

        # Output conv
        self.out_conv = nn.Conv2d(base_filters, out_channels, 1)

    def forward(self, x):
        identity = x  # residual learning: predict streaks

        # Encoder
        s1, p1 = self.d1(x)
        s2, p2 = self.d2(p1)
        s3, p3 = self.d3(p2)
        s4, p4 = self.d4(p3)

        # Bottleneck
        b = self.bottleneck(p4)

        # Decoder
        x = self.u4(b, s4)
        x = self.u3(x, s3)
        x = self.u2(x, s2)
        x = self.u1(x, s1)

        # Predict streak artifacts
        streaks = self.out_conv(x)

        # Subtract predicted streaks to get clean slice
        return identity - streaks

In [34]:
from torch.utils.data import DataLoader, TensorDataset
def predict_streak_free_volume(model, volume, batch_size=2, device='cuda'):
    """
    Predicts a streak-free volume from a 3D CT volume using a trained U-Net.

    Args:
        model: trained U-Net model (residual U-Net preferred)
        volume: NumPy array of shape (N_slices, H, W)
        batch_size: batch size for inference
        device: 'cuda' or 'cpu'

    Returns:
        cleaned_volume: NumPy array of same shape as input (N, H, W)
    """
    model.to(device)
    model.eval()
    
    # Normalize and convert to tensor
    vol_tensor = torch.tensor(volume.astype('float32') / volume.max()).unsqueeze(1).to(device)  # (N,1,H,W)
    
    dataset = TensorDataset(vol_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    cleaned_slices = []
    with torch.no_grad():
        for (x_batch,) in loader:
            out = model(x_batch)  # residual U-Net: outputs corrected slice
            cleaned_slices.append(out.cpu().numpy())
    
    # Concatenate and remove channel dimension
    cleaned_volume = np.concatenate(cleaned_slices, axis=0)
    cleaned_volume = cleaned_volume.squeeze(1)  # shape: (N,H,W)
    
    # Optional: rescale to original intensity range
    cleaned_volume = (cleaned_volume * volume.max()).astype(volume.dtype)
    
    return cleaned_volume


In [52]:
model = StreakResUNet()
model.load_state_dict(torch.load("resnet_epoch_20.pth"))

<All keys matched successfully>

In [53]:
scan = 'L008'
alt_scan = read_alt_scan(scan)
raw_scan = read_raw_scan(scan)

In [54]:
new_scan = predict_streak_free_volume(model, alt_scan)

In [55]:
combo = create_side_by_side(alt_scan, new_scan)
combo = create_side_by_side(combo, raw_scan)

In [56]:
sv.slice(combo, slice_number=0, continuous_update=1)

In [57]:
alt_new_diff = alt_scan - new_scan
raw_new_diff = raw_scan - new_scan
diff_combo = create_side_by_side(alt_new_diff, raw_new_diff)

In [58]:
sv.slice(diff_combo, slice_number=0, continuous_update=1)

In [21]:
write_manipulated_to_file(new_scan, scan)

100%|██████████| 110/110 [00:00<00:00, 2761.05it/s]


In [42]:
for i in tqdm(range(len(alt_scan_dirs))):
    scan = alt_scan_dirs[i]
    alt_scan = read_alt_scan(scan)
    new_scan = predict_streak_free_volume(model, alt_scan)
    write_manipulated_to_file(new_scan, scan)

100%|██████████| 20/20 [02:17<00:00,  6.86s/it]
